# Python Automation & ETL Basics: The Messy Grocery List
**Date:** 07/09/2026

This notebook demonstrates a basic ETL (Extract, Transform, Load) pipeline. 
We will take a messy`.txt` grocery list, clean the data using `pandas`, and store the final output in an Excel spreadsheet.


### Step 1. Install the necessary packages

In [1]:
!pip install openpyxl
!mamba install pandas

mambajs 0.21.4

Process pip requirements ...

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 2.171 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.0.4    np23py313h1e705a5_0  emscripten-forge-4x
+ python-tzdata  2026.2   pyhd8ed1ab_0         conda-forge
- pip            26.1.2   pyh145f28c_0         conda-forge


### Step 2. Import necessary packages

In [2]:
import pandas as pd # extract, transform
import logging
import os

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    handlers=[
        logging.FileHandler("etl_pipeline.log", mode='w'),
        logging.StreamHandler()                  
    ]
)

### Step 3. Extract. Assign it to a variable called messy_grocery_list
Fetching raw data from our source file. We will read the `messy_grocery_list.txt` file into a pandas DataFrame so we can easily manipulate it.

In [4]:
def extract_data(filepath):
    """Reads a text file and loads it into a pandas DataFrame."""
    messy_grocery_list = pd.read_csv(filepath, sep="-", header=None, names=["Item_Raw","Details_Raw"])
    return messy_grocery_list

In [5]:
file_path = 'messy_grocery_list.txt'
messy_grocery_list = extract_data(file_path)
logging.info(f"Extraction complete. {len(messy_grocery_list)} rows loaded.")

2026-07-10 17:28:23,363 - INFO - Extraction complete. 5 rows loaded.


### Step 4. Display the raw contents of the messy grocery list.

In [6]:
messy_grocery_list

,Item_Raw,Details_Raw
0,apples,1 pieces
1,MILK!!!,2 cartons
2,banana,6 yellow banana
3,juicE,1 liter
4,MILK!!!,2 cartons


### Step 5. Transform.
This is where `pandas` shines. We will clean the data by:
1. Cleaning the item names (removing special characters like `!!!`, standardizing to lowercase, stripping extra spaces, and removing duplicates).
2. Extracting the numerical value (quantity) and the unit of measurement into separate columns.

In [7]:
# def transform_data(messy_grocery_list):
#     """Cleans and structures the raw dataframe."""
#     logging.info("Starting data transformation...")
#     try:
#         logging.info(messy_grocery_list[['Item_Raw', 'Details_Raw']].head())
        
#         # 1. Clean the 'Item_Raw' column
#         # Convert to lowercase, remove everything except letters (a-z), and strip edge spaces
#         messy_grocery_list['Item'] = messy_grocery_list['Item_Raw'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()
        
#         # 2. Clean the 'Details_Raw' column to extract Quantity and Unit
#         # Extract the first number found as Quantity
#         messy_grocery_list['Quantity'] = messy_grocery_list['Details_Raw'].str.extract(r'(\d+)').astype(float)
        
#         # Extract the alphabetical characters after the number as the Unit
#         messy_grocery_list['Unit'] = messy_grocery_list['Details_Raw'].str.replace(r'\d+', '', regex=True).str.strip()
        
#         # 3. Filter down to only the clean columns we want
#         clean_grocery_list = messy_grocery_list[['Item', 'Quantity', 'Unit']].copy()

#         # 4. Remove duplicated rows
#         clean_grocery_list = clean_grocery_list.drop_duplicates(ignore_index=True)
        
#         logging.info("Data transformation successful.")
#         return clean_grocery_list
        
#     except Exception as e:
#         # Error handling allows the script to fail gracefully
#         logging.error(f"An error occurred during transformation: {e}")
#         raise

In [8]:
# 1. Clean the 'Item_Raw' column
# Convert to lowercase, remove everything except letters (a-z), and strip edge spaces
messy_grocery_list['Item'] = messy_grocery_list['Item_Raw'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()
messy_grocery_list

,Item_Raw,Details_Raw,Item
0,apples,1 pieces,apples
1,MILK!!!,2 cartons,milk
2,banana,6 yellow banana,banana
3,juicE,1 liter,juice
4,MILK!!!,2 cartons,milk


In [9]:
# 2. Clean the 'Details_Raw' column to extract Quantity and Unit
# Extract the first number found as Quantity
messy_grocery_list['Quantity'] = messy_grocery_list['Details_Raw'].str.extract(r'(\d+)').astype(int)
messy_grocery_list

,Item_Raw,Details_Raw,Item,Quantity
0,apples,1 pieces,apples,1
1,MILK!!!,2 cartons,milk,2
2,banana,6 yellow banana,banana,6
3,juicE,1 liter,juice,1
4,MILK!!!,2 cartons,milk,2


In [10]:
# Extract the alphabetical characters after the number as the Unit
messy_grocery_list['Unit'] = messy_grocery_list['Details_Raw'].str.replace(r'\d+', '', regex=True).str.strip()
messy_grocery_list

,Item_Raw,Details_Raw,Item,Quantity,Unit
0,apples,1 pieces,apples,1,pieces
1,MILK!!!,2 cartons,milk,2,cartons
2,banana,6 yellow banana,banana,6,yellow banana
3,juicE,1 liter,juice,1,liter
4,MILK!!!,2 cartons,milk,2,cartons


In [11]:
# 3. Filter down to only the clean columns we want
cleaned_grocery_list = messy_grocery_list[['Item', 'Quantity', 'Unit']].copy()
cleaned_grocery_list

,Item,Quantity,Unit
0,apples,1,pieces
1,milk,2,cartons
2,banana,6,yellow banana
3,juice,1,liter
4,milk,2,cartons


In [12]:
# 4. Remove duplicated rows
cleaned_grocery_list = cleaned_grocery_list.drop_duplicates(ignore_index=True)
cleaned_grocery_list

,Item,Quantity,Unit
0,apples,1,pieces
1,milk,2,cartons
2,banana,6,yellow banana
3,juice,1,liter


In [13]:
logging.info("Data transformation successful.")

2026-07-10 17:28:23,576 - INFO - Data transformation successful.


In [14]:
# Transform
#cleaned_grocery_list = transform_data(messy_grocery_list)

### Step 6. Display the contents of the cleaned grocery list.

In [15]:
cleaned_grocery_list

,Item,Quantity,Unit
0,apples,1,pieces
1,milk,2,cartons
2,banana,6,yellow banana
3,juice,1,liter


### Step 7. Load
Storing the processed data into its final destination. We will save this to a **clean Excel spreadsheet**.

In [16]:
def load_data(cleaned_grocery_list, excel_filename):
    """Saves the cleaned grocery list to Excel."""
    logging.info("Starting load process...")
    try:
        # --- LOAD TO EXCEL ---
        cleaned_grocery_list.to_excel(excel_filename, index=False, engine='openpyxl')
        logging.info(f"Successfully loaded data into Excel: {excel_filename}")
        
    except Exception as e:
        logging.error(f"Failed to load data: {e}")
        raise

In [17]:
# Load
# cleaned_grocery_list = transform_data(messy_grocery_list)
excel_filename = 'cleaned_grocery_list.xlsx'
load_data(cleaned_grocery_list, excel_filename)

2026-07-10 17:28:23,672 - INFO - Starting load process...
2026-07-10 17:28:24,304 - INFO - Successfully loaded data into Excel: cleaned_grocery_list.xlsx


In [18]:
logging.info("Verifying Excel load...")

try:
    cleaned_list_excel = pd.read_excel("cleaned_grocery_list.xlsx", index_col=None)
    logging.info("Pipeline executed successfully! Here is the final loaded data:")
    
except Exception as e:
    logging.error(f"Verification failed: {e}")

2026-07-10 17:28:24,321 - INFO - Verifying Excel load...
2026-07-10 17:28:24,463 - INFO - Pipeline executed successfully! Here is the final loaded data:


In [19]:
cleaned_list_excel

,Item,Quantity,Unit
0,apples,1,pieces
1,milk,2,cartons
2,banana,6,yellow banana
3,juice,1,liter


In [20]:
# 1. Add a Price column 
# (In a real scenario, we might extract this from a database, but for the demo we'll hardcode it)
cleaned_grocery_list['Price'] = [10.00, 50.50, 12.30, 24.75]
cleaned_grocery_list

,Item,Quantity,Unit,Price
0,apples,1,pieces,10.00
1,milk,2,cartons,50.50
2,banana,6,yellow banana,12.30
3,juice,1,liter,24.75


In [21]:
# 2. Multiply Quantity by Price to create the 'Amount' column
cleaned_grocery_list['Amount'] = cleaned_grocery_list['Quantity'] * cleaned_grocery_list['Price']
cleaned_grocery_list

,Item,Quantity,Unit,Price,Amount
0,apples,1,pieces,10.00,10.00
1,milk,2,cartons,50.50,101.00
2,banana,6,yellow banana,12.30,73.80
3,juice,1,liter,24.75,24.75


In [22]:
# 3. Calculate the Total Amount
total_amount = cleaned_grocery_list['Amount'].sum()

# We create a new row with the total and append it.
new_row = pd.DataFrame({'Item': ['TOTAL'], 'Quantity': [''], 'Unit': [''], 'Price': [''], 'Amount': [total_amount]})
cleaned_grocery_list = pd.concat([cleaned_grocery_list, new_row], ignore_index=True)
cleaned_grocery_list

,Item,Quantity,Unit,Price,Amount
0,apples,1,pieces,10.0,10.00
1,milk,2,cartons,50.5,101.00
2,banana,6,yellow banana,12.3,73.80
3,juice,1,liter,24.75,24.75
4,TOTAL,,,,209.55


In [23]:
# 4. Load: Export the final, fully calculated table to Excel
cleaned_grocery_list.to_excel("updated_grocery_list.xlsx", index=False)
logging.info("Updated grocery list with prices exported to Excel.")

2026-07-10 17:28:24,676 - INFO - Updated grocery list with prices exported to Excel.


In [24]:
# Display the final dataframe in Jupyter
cleaned_grocery_list

,Item,Quantity,Unit,Price,Amount
0,apples,1,pieces,10.0,10.00
1,milk,2,cartons,50.5,101.00
2,banana,6,yellow banana,12.3,73.80
3,juice,1,liter,24.75,24.75
4,TOTAL,,,,209.55


In [25]:
logging.info(f"The total grocery bill is: Php{total_amount:.2f}")

2026-07-10 17:28:24,716 - INFO - The total grocery bill is: Php209.55


In [26]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    handlers=[
        logging.FileHandler("etl_pipeline.log", mode='w'),                
    ]
)